In [0]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Cargar los datos de la capa GOLD
data = spark.table("gold_nvda_features").toPandas()

# 2. Seleccionar nuestras variables (Features) y lo que queremos predecir (Target)
features = ["open_price", "high_price", "low_price", "close_price", "volume", "sma_7", "sma_30", "daily_volatility"]
X = data[features]
y = data["target"]

# 3. Dividir los datos: 80% para entrenar, 20% para evaluar
# IMPORTANTE: En series de tiempo no se baraja (shuffle=False) porque el orden importa
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f"Set de entrenamiento: {X_train.shape[0]} días")
print(f"Set de prueba: {X_test.shape[0]} días")

In [0]:
from mlflow.models import infer_signature

# 1. Definir el nombre del experimento (puedes cambiarlo si quieres)
mlflow.set_experiment("/Users/" + spark.sql("SELECT current_user()").collect()[0][0] + "/nvda_prediction_experiment")

with mlflow.start_run(run_name="RandomForest_Basico"):
    # 2. Configurar el modelo
    # n_estimators=100 significa que usaremos 100 árboles de decisión
    n_trees = 100
    model = RandomForestClassifier(n_estimators=n_trees, random_state=42)
    
    # 3. ENTRENAR
    model.fit(X_train, y_train)
    
    # 4. PREDECIR Y EVALUAR
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    
    # 5. CREAR SIGNATURE E INPUT_EXAMPLE (requerido para Unity Catalog)
    signature = infer_signature(X_train, predictions)
    input_example = X_train[:5]  # Primeras 5 filas como ejemplo
    
    # 6. REGISTRAR EN MLFLOW
    # Guardamos los parámetros que elegimos
    mlflow.log_param("n_estimators", n_trees)
    # Guardamos el resultado del modelo
    mlflow.log_metric("accuracy", accuracy)
    # Guardamos el modelo físico para usarlo después - CON SIGNATURE E INPUT_EXAMPLE
    mlflow.sklearn.log_model(
        model, 
        "model_nvda",
        signature=signature,
        input_example=input_example
    )
    
    print(f"Modelo entrenado con éxito. Precisión (Accuracy): {accuracy:.2%}")

In [0]:
# 1. Obtener el ID de la última corrida que acabas de hacer
run_id = mlflow.last_active_run().info.run_id

# 2. Usar el nombre completo de Unity Catalog (catalog.schema.model_name)
model_name = "workspace.default.nvda_price_classifier"

# 3. Registrar el modelo en Unity Catalog
# Esto lo mueve de ser un "archivo" a ser un "objeto de negocio"
model_uri = f"runs:/{run_id}/model_nvda"
reg_model = mlflow.register_model(model_uri, model_name)

print(f"Modelo registrado como: {model_name}")
print(f"Versión actual: {reg_model.version}")